# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: Shiv Mahesh Bhosale
#### Student ID: S4192953


Environment: Python 397 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
You should give a brief information of this assessment task here.

<span style="color: red"> Note that this is a sample notebook only. You will need to fill in the proper markdown and code blocks. You might also want to make necessary changes to the structure to meet your own needs. Note also that any generic comments written in this notebook are to be removed and replace with your own words.</span>

## Importing libraries 

In [33]:
# import pandas as pd
# import numpy as np
# import regex as re
from collections import Counter
from itertools import chain
import gensim
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import lil_matrix
import warnings
import logging

import gensim.downloader as api
warnings.filterwarnings('ignore')
import os

In [2]:
!pip install pandas numpy scikit-learn gensim nltk

In [3]:
import sys

print(sys.executable)
print(sys.version)

C:\Users\shivb\anaconda3\envs\py397\python.exe
3.9.25 (main, Nov  3 2025, 22:44:01) [MSC v.1929 64 bit (AMD64)]


In [4]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn gensim nltk

In [42]:
import pandas as pd
import numpy as np
from collections import Counter
from itertools import chain
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import gensim.downloader as api
import urllib.request
import zipfile

## Task 2. Generating Feature Representations for Clothing Items Reviews

...... Sections and code blocks on buidling different document feature represetations


<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [6]:
def load_vocab(filepath='vocab'):
    vocab = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                word, idx = line.rsplit(':', 1)
                vocab[word] = int(idx)
    return vocab

In [7]:
vocab = load_vocab('vocab.txt')
index_to_word = {idx: word for word, idx in vocab.items()}

In [8]:
print(f"Vocabulary size: {len(vocab)}")
print(f"\nFirst 10 vocab entries:")
for word, idx in list(vocab.items())[:10]:
    print(f"  {word}:{idx}")

Vocabulary size: 8018

First 10 vocab entries:
  a-cup:0
  a-flutter:1
  a-frame:2
  a-kind:3
  a-line:4
  a-lines:5
  a-symmetric:6
  aa:7
  ab:8
  abbey:9


In [9]:
df = pd.read_csv('processed.csv')

In [10]:
df

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,had such high hopes for this dress and really ...,3,0,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,love love love this jumpsuit it's fun flirty a...,5,1,0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,this shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
3,1080,49,Not for the very petite,love tracy reese dresses but this one is not f...,2,0,4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,this in my basket at hte last to see what it w...,5,1,1,General Petite,Tops,Knits
...,...,...,...,...,...,...,...,...,...,...
19657,1104,34,Great dress for many occasions,was very happy to snag this dress at such grea...,5,1,0,General Petite,Dresses,Dresses
19658,862,48,Wish it was made of cotton,it reminds me of maternity clothes soft stretc...,3,1,0,General Petite,Tops,Knits
19659,1104,31,"Cute, but see through",this fit well but the top was very see through...,3,0,1,General Petite,Dresses,Dresses
19660,1084,28,"Very cute dress, perfect for summer parties an...",bought this dress for wedding have this summer...,3,1,2,General,Dresses,Dresses


In [11]:
df["Review Text"].isnull().sum()

np.int64(0)

In [12]:
tk_reviews = [review.split(' ') for review in df['Review Text'].tolist()]

In [13]:
tk_reviews[1]

['love',
 'love',
 'love',
 'this',
 'jumpsuit',
 "it's",
 'fun',
 'flirty',
 'and',
 'fabulous',
 'every',
 'time',
 'wear',
 'it',
 'get',
 'nothing',
 'but',
 'great',
 'compliments']

In [14]:
joined_reviews = [' '.join(tokens) for tokens in tk_reviews]

In [15]:
joined_reviews[1]

"love love love this jumpsuit it's fun flirty and fabulous every time wear it get nothing but great compliments"

In [16]:
len(tk_reviews)

19662

In [17]:
cVectorizer = CountVectorizer(analyzer="word", vocabulary=vocab)
count_features = cVectorizer.fit_transform(joined_reviews)

In [18]:
count_features.shape

(19662, 8018)

In [19]:
out_file = open('count_vectors.txt', 'w') 

In [20]:
for doc_id in range(count_features.shape[0]):
    row = count_features[doc_id].tocoo()
    pairs = ','.join(
        f"{col}:{val}"
        for col, val in sorted(zip(row.col, row.data))
    )
    out_file.write(f"#{doc_id},{pairs}\n")
out_file.close()

In [21]:
with open('count_vectors.txt') as f:
    lines = f.read().splitlines()

In [22]:
len(lines)

19662

In [23]:
lines[0]

'#0,227:3,516:1,757:1,923:2,1112:1,1332:1,1504:1,1820:1,1900:1,2061:1,2427:1,2579:1,2627:1,2721:2,2754:1,3056:2,3062:2,3197:1,3277:1,3392:1,3414:2,3462:1,3518:1,3553:4,3623:1,3775:2,3778:1,4071:1,4156:1,4179:1,4423:1,4487:2,4502:1,4548:1,4627:1,4693:1,4726:1,4735:2,4742:1,4947:2,5446:1,5584:1,6007:1,6014:1,6218:1,6315:3,6371:1,6411:1,6756:1,7028:1,7033:5,7084:2,7135:1,7169:2,7195:1,7396:1,7490:1,7525:1,7579:1,7647:1,7669:3,7782:1,7886:1,8009:1,8011:1'

In [24]:
tVectorizer = TfidfVectorizer(analyzer="word", vocabulary=vocab)
tfidf_features = tVectorizer.fit_transform(joined_reviews)

In [25]:
tfidf_features.shape

(19662, 8018)

In [30]:
sample = tfidf_features[0].tocoo()
top = sorted(zip(sample.col, sample.data), key=lambda x: -x[1])[:8]

print("Top TF-IDF words in review 0:")
for index, score in top:
    print(f"  '{index_to_word[int(index)]}': {score:.4f}")

Top TF-IDF words in review 0:
  'net': 0.4086
  'half': 0.2517
  'layer': 0.2299
  'outrageously': 0.2155
  'directly': 0.1850
  'small': 0.1840
  'over': 0.1704
  'reordered': 0.1688


In [39]:
if not os.path.exists('glove/glove.6B.50d.txt'):
    print("Downloading GloVe...")
    urllib.request.urlretrieve(
        "https://titan.csit.rmit.edu.au/~e28347/glove.6B.50d.txt.zip",
        "glove.6B.50d.txt.zip"
    )
    with zipfile.ZipFile("glove.6B.50d.txt.zip", 'r') as z:
        z.extractall("glove")
    print("✅ GloVe ready")
else:
    print("GloVe already downloaded ✅")

HTTPError: HTTP Error 404: Not Found

In [40]:
if not os.path.exists('glove/glove.6B.50d.txt'):
    print("Downloading GloVe...")
    urllib.request.urlretrieve(
        "https://titan.csit.rmit.edu.au/~e28347/glove.6B.50d.txt.zip",
        "glove.6B.50d.txt.zip"
    )
    with zipfile.ZipFile("glove.6B.50d.txt.zip", 'r') as z:
        z.extractall("glove")
    print("GloVe ready")
else:
    print("GloVe already downloaded ")

HTTPError: HTTP Error 404: Not Found

In [41]:
import os
import numpy as np

# path_to_glove_file = os.path.join("glove/glove.6B.100d.txt") # specified the path to the embedding file
path_to_glove_file = os.path.join("glove/glove.6B.50d.txt") # specified the path to the embedding file

glove_wv = {} # initialise an empty dcitionary
with open(path_to_glove_file, encoding="utf-8") as f: # open the txt file containing the word embedding vectors
    for line in f:
        word, coefs = line.split(maxsplit=1) # The maxsplit defines the maximum number of splits. 
                                             # in the above example, it will give:
                                             # ['population','0.035182 1.4248 0.9758 0.1313 -0.66877 0.8539 -0.11525 ......']
        coefs = np.fromstring(coefs, "f", sep=" ") # construct an numpy array from the string 'coefs', 
                                                   # e.g., '0.035182 1.4248 0.9758 0.1313 -0.66877 0.8539 -0.11525 ......'
        glove_wv[word] = coefs # create the word and embedding vector mapping

print("Found %s word vectors." % len(glove_wv))

FileNotFoundError: [Errno 2] No such file or directory: 'glove/glove.6B.50d.txt'

### Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [27]:
# code to save output data...

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [28]:
# Code to perform the task...


## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>